# Interception wet-leaf-temperature loop: does it need relaxation? (debug/tl-convergence-isolation)

**Research/debug notebook -- lives on branch `debug/tl-convergence-isolation`, not intended to be merged.**

`Interception.run()`'s inner wet-leaf energy-balance loop
(`pyAPES/canopy/interception.py`, the `while err > 0.01 and iterNo < itermax:` block around
lines 176-211) solves `Tl_wet` by plain fixed-point iteration -- unlike the outer mlm_canopy
Picard loop (`pyAPES/canopy/mlm_canopy.py`), it applies **no relaxation** to the update at all.
This notebook checks whether that matters: it copies the loop into a standalone sandbox
function (same pattern as `debug_mlm_canopy_convergence.ipynb`'s `picard_loop`), adds a
`gamma` relaxation knob (and an optional oscillation-adaptive `gamma_floor`, mirroring
`mlm_canopy.py`'s own `gam = max(gam / 2, floor)` trick), and sweeps it across every timestep's
forcing already captured in `debug_captures/forcing_exploration/interception_forcing_samples.pkl`
(1392 timesteps, 1.6.-30.6.2022, captured at `iter_no==2` -- see
`debug_forcing_exploration.ipynb` for how/why). **No new capture run is needed.**

This notebook does **not** modify `pyAPES/canopy/interception.py`. If relaxation reliably
helps here, the minimal equivalent change should be ported into the real source afterwards,
then re-verified.

**Caveat on the initial guess for `Tl_wet`:** in the real model, `CanopyModel._restore()` sets
`self.interception.Tl_wet = air_temperature` once at the very start of each timestep (before
the outer Picard loop runs); on later outer iterations `Tl_wet` is warm-started from the
previous outer iteration's converged value. The captured forcing here is a snapshot from
`iter_no==2`, but the intermediate `Tl_wet` warm-start itself wasn't captured, so this notebook
initializes `Tl_wet0 = air_temperature` for every sample. That's exactly what the real model
does on a timestep's first outer iteration, and is a reasonable, clearly-flagged simplification
for isolating the inner loop's own convergence behaviour -- just keep in mind that on later
outer iterations the real warm start may differ.

In [ ]:
import os
import sys

# Jupyter runs this notebook's kernel with cwd = this file's own directory (Examples/),
# same convention as the other debug notebooks on this branch.
assert os.path.basename(os.getcwd()) == 'debug', (
    f"expected to run with cwd=debug/ (this notebook's own directory), got {os.getcwd()!r} -- adjust paths below if not")
sys.path.insert(0, os.path.abspath('..'))  # repo root, so `import pyAPES` works

import pickle
import copy
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt

FORCING_CAPTURE_DIR = '../Examples/debug_captures/forcing_exploration'
CAPTURE_FILE = os.path.join(FORCING_CAPTURE_DIR, 'interception_forcing_samples.pkl')

with open(CAPTURE_FILE, 'rb') as f:
    records = pickle.load(f)

print(f'{len(records)} captured interception-forcing records loaded from {CAPTURE_FILE}')
outcome_counts = pd.Series([r['outcome'] for r in records]).value_counts()
print('outer mlm_canopy outcome breakdown:')
print(outcome_counts)

In [ ]:
# LAIz and leaf_length are static structural attributes of CanopyModel (not part of the
# per-timestep forcing dict, so they weren't captured). Build one real CanopyModel instance
# to get them -- this is cheap: object construction only, no simulation timestepping.
from pyAPES.soil.soil import Soil_1D
from pyAPES.canopy.mlm_canopy import CanopyModel
from pyAPES.parameters.mlm_parameters_FI_Ran import gpara, cpara, spara

soil = Soil_1D(spara)
canopy_model = CanopyModel(cpara, dz_soil=soil.grid['dz'])

LAIz = canopy_model.lad * canopy_model.dz
leaf_length = canopy_model.leaf_length

print('LAIz shape:', LAIz.shape, '| nonzero layers:', int(np.sum(LAIz > 0)))
print('leaf_length shape:', leaf_length.shape)
print('sample record array shape:', records[0]['air_temperature'].shape)
assert LAIz.shape == records[0]['air_temperature'].shape, 'LAIz layering does not match captured forcing arrays'

## EXPERIMENTAL sandbox -- do not treat as source of truth

The cell below is a **copy** of the wet-leaf energy-balance loop from `Interception.run()`
(`pyAPES/canopy/interception.py`, roughly lines 172-211), with a relaxation factor `gamma`
added to the update:

```
Tl_wet_candidate = <same fixed-point update as the source>
Tl_wet = Told + gamma * (Tl_wet_candidate - Told)
```

`gamma=1.0` reproduces the unmodified source exactly (no relaxation) -- that's the baseline.
An optional `gamma_floor` / `osc_check_after` pair adds the same oscillation-adaptive damping
`mlm_canopy.py` already uses for its outer loop (`mlm_canopy.py:589-592`): if the squared error
grew relative to two iterations back, average the last two `Tl_wet` values and halve `gamma`
down to `gamma_floor`. Leave `gamma_floor=None` to disable that and use a plain fixed `gamma`.

**This copy will drift from the source over time.** Any fix validated here must be ported back
into `interception.py` as a real patch, then re-verified.

In [ ]:
from pyAPES.leaf.boundarylayer import leaf_boundary_layer_conductance
from pyAPES.microclimate.micromet import e_sat, latent_heat
from pyAPES.utils.constants import MOLAR_MASS_H2O, SPECIFIC_HEAT_AIR, EPS

EPS_MACHINE = np.finfo(float).eps


def solve_wet_leaf_temperature(record, LAIz, leaf_length, gamma=1.0, gamma_floor=None,
                                osc_check_after=None, max_iter=30, max_err=0.01):
    # Sandbox copy of Interception.run()'s wet-leaf energy-balance loop. See markdown above.
    P = record['air_pressure']
    Tl_ave = record['leaf_temperature']
    H2O = record['h2o']
    U = record['wind_speed']
    T = record['air_temperature']
    gr = record['lw_radiative_conductance']
    Rabs = record['sw_absorbed'] + record['net_lw_leaf']

    lt = np.maximum(EPS, leaf_length)
    N = len(LAIz)
    ic = np.where(LAIz > 0)

    # initial guess: air temperature, matching CanopyModel._restore() cold start (see markdown)
    Tl_wet = T.copy()
    Told = Tl_wet.copy()

    L = latent_heat(T) * MOLAR_MASS_H2O
    es, s = e_sat(Tl_wet)
    Dleaf = es / P - H2O
    s = s / P

    gam = gamma
    traj = []
    err = 999.0
    iterNo = 0
    while err > max_err and iterNo < max_iter:
        iterNo += 1

        gb_h, _, gb_v = leaf_boundary_layer_conductance(U, lt, T, 0.5 * (Tl_wet + Told) - T, P)
        Told = Tl_wet.copy()

        Tl_wet_candidate = Tl_wet.copy()
        Tl_wet_candidate[ic] = (
            Rabs[ic] + SPECIFIC_HEAT_AIR * gr[ic] * Tl_ave[ic] + SPECIFIC_HEAT_AIR * gb_h[ic] * T[ic]
            - L[ic] * gb_v[ic] * Dleaf[ic] + L[ic] * s[ic] * gb_v[ic] * Told[ic]
        ) / (SPECIFIC_HEAT_AIR * (gr[ic] + gb_h[ic]) + L[ic] * s[ic] * gb_v[ic])

        Tl_wet = Told.copy()
        Tl_wet[ic] = Told[ic] + gam * (Tl_wet_candidate[ic] - Told[ic])
        err = np.nanmax(np.abs(Tl_wet - Told))

        # oscillation-adaptive damping, mirrors mlm_canopy.py:589-592
        if gamma_floor is not None and osc_check_after is not None and iterNo > osc_check_after:
            prev_err = traj[-2]['err'] if len(traj) >= 2 else None
            if prev_err is not None and err > prev_err:
                Tl_wet[ic] = 0.5 * (Told[ic] + Tl_wet[ic])
                gam = max(gam / 2, gamma_floor)
                err = np.nanmax(np.abs(Tl_wet - Told))

        es, s = e_sat(Tl_wet)
        Dleaf = es / P - H2O
        s = s / P

        traj.append({'iter': iterNo, 'err': err, 'gam': gam, 'Tl_wet_mean': float(np.nanmean(Tl_wet[ic]))})

    converged = err <= max_err and iterNo < max_iter
    return {'traj': traj, 'iterNo': iterNo, 'err': err, 'converged': converged, 'Tl_wet': Tl_wet}

In [ ]:
def sweep(records, LAIz, leaf_length, **kwargs):
    rows = []
    for r in records:
        res = solve_wet_leaf_temperature(r, LAIz, leaf_length, **kwargs)
        rows.append({
            'timestamp': r['timestamp'],
            'hour': r['timestamp'].hour,
            'outer_outcome': r['outcome'],
            'iterNo': res['iterNo'],
            'final_err': res['err'],
            'converged': res['converged'],
        })
    return pd.DataFrame(rows)


baseline_df = sweep(records, LAIz, leaf_length, gamma=1.0, gamma_floor=None, max_iter=30, max_err=0.01)

n_fail = int((~baseline_df['converged']).sum())
print(f"baseline (gamma=1.0, i.e. unmodified source): {n_fail}/{len(baseline_df)} "
      f"interception wet-leaf solves fail to converge within 30 iterations")
print()
print('breakdown of baseline non-convergence by outer mlm_canopy outcome:')
print(baseline_df.loc[~baseline_df['converged'], 'outer_outcome'].value_counts())
baseline_df.sort_values('final_err', ascending=False).head(10)

In [ ]:
COLOR_CONVERGED = '#0ca30c'
COLOR_NONCONVERGED = '#d03b3b'

order = baseline_df.sort_values('final_err', ascending=False).reset_index(drop=True)
colors = [COLOR_CONVERGED if c else COLOR_NONCONVERGED for c in order['converged']]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(order)), order['final_err'], color=colors, width=1.0)
ax.axhline(0.01, color='k', ls='--', lw=0.8, label='max_err')
ax.set_yscale('log')
ax.set_xlabel('captured timestep (sorted by final error)')
ax.set_ylabel('final |Tl_wet - Told| (log scale)')
ax.set_title(f'baseline (gamma=1.0): {len(order) - n_fail}/{len(order)} converged')
ax.legend()
plt.tight_layout()
plt.show()

## Try adding relaxation

Edit `GAMMA` (and optionally `GAMMA_FLOOR` / `OSC_CHECK_AFTER` to also enable oscillation-adaptive
damping) below and re-run this section to try different values against all 1392 captured
timesteps at once.

In [ ]:
# --- user-adjustable knobs ---
# updated to match the parameters actually ported into pyAPES/canopy/interception.py
# (debug/tl-convergence-isolation fix, incl. the prev_err/gamma bugfix)
GAMMA = 0.75
GAMMA_FLOOR = 0.01   # enables oscillation-adaptive damping, matches source
OSC_CHECK_AFTER = 5
MAX_ITER = 30
MAX_ERR = 0.01

relaxed_df = sweep(records, LAIz, leaf_length, gamma=GAMMA, gamma_floor=GAMMA_FLOOR,
                    osc_check_after=OSC_CHECK_AFTER, max_iter=MAX_ITER, max_err=MAX_ERR)

n_fail_relaxed = int((~relaxed_df['converged']).sum())
print(f"relaxed (gamma={GAMMA}, gamma_floor={GAMMA_FLOOR}): "
      f"{n_fail_relaxed}/{len(relaxed_df)} fail to converge within {MAX_ITER} iterations")
relaxed_df.sort_values('final_err', ascending=False).head(10)


In [ ]:
summary = pd.DataFrame({
    'baseline (gamma=1.0)': [n_fail, len(baseline_df) - n_fail,
                              baseline_df['iterNo'].mean(), baseline_df['iterNo'].median()],
    f'relaxed (gamma={GAMMA}, floor={GAMMA_FLOOR})': [n_fail_relaxed, len(relaxed_df) - n_fail_relaxed,
                              relaxed_df['iterNo'].mean(), relaxed_df['iterNo'].median()],
}, index=['n_non_converged', 'n_converged', 'mean_iterations', 'median_iterations'])
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(baseline_df['iterNo'], bins=range(1, MAX_ITER + 2), alpha=0.6, label='baseline (gamma=1.0)', color=COLOR_NONCONVERGED)
ax.hist(relaxed_df['iterNo'], bins=range(1, MAX_ITER + 2), alpha=0.6, label=f'relaxed (gamma={GAMMA})', color=COLOR_CONVERGED)
ax.set_xlabel('iterations to converge (or max_iter if not converged)')
ax.set_ylabel('count of timesteps')
ax.legend()
ax.set_title('iterations-to-converge distribution')

# worst-offending timestep under baseline
worst_idx = baseline_df['final_err'].idxmax()
worst_record = records[worst_idx]
res_baseline = solve_wet_leaf_temperature(worst_record, LAIz, leaf_length, gamma=1.0, max_iter=MAX_ITER, max_err=MAX_ERR)
res_relaxed = solve_wet_leaf_temperature(worst_record, LAIz, leaf_length, gamma=GAMMA, gamma_floor=GAMMA_FLOOR,
                                          osc_check_after=OSC_CHECK_AFTER, max_iter=MAX_ITER, max_err=MAX_ERR)

ax = axes[1]
ax.plot(range(1, len(res_baseline['traj']) + 1), [s['err'] for s in res_baseline['traj']],
        marker='o', ms=3, color=COLOR_NONCONVERGED, label='baseline (gamma=1.0)')
ax.plot(range(1, len(res_relaxed['traj']) + 1), [s['err'] for s in res_relaxed['traj']],
        marker='o', ms=3, color=COLOR_CONVERGED, label=f'relaxed (gamma={GAMMA})')
ax.axhline(MAX_ERR, color='k', ls='--', lw=0.8, label='max_err')
ax.set_yscale('log')
ax.set_xlabel('iteration')
ax.set_ylabel('|Tl_wet - Told|')
ax.set_title(f"worst baseline timestep: {worst_record['timestamp']} ({worst_record['outcome']})")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Next steps

If a `GAMMA` (and optionally `GAMMA_FLOOR`) value above reliably reduces iterations-to-converge
and non-convergence count across the full 1392-timestep set, port the equivalent minimal change
into the real `while err > 0.01 and iterNo < itermax:` loop in
`Interception.run()` (`pyAPES/canopy/interception.py`, lines ~176-211) -- not this notebook.
After doing so, re-run `Examples/capture_forcing_exploration.py` and check whether the number of
timesteps where interception hits its own `itermax` (currently only visible as a debug log
message) drops.